In [10]:
using Pkg                # Load Julia's built-in package manager module (Pkg)

Pkg.activate("..")       # Activate the Project.toml located one directory above
                         # (Allows notebooks/ to use the repository root environment)

using Revise             # Development tool that automatically reloads changes
                         # made to files in src/ without restarting Julia

using Horizons3DViewer2  # Load the custom package Horizons3DViewer2
                         # (This loads src/Horizons3DViewer2.jl and its modules)

using EphemerisSources   # Tools for accessing planetary ephemeris data
using AstroTime, Dates   # Time systems and date handling utilities
using GLMakie            # High-performance interactive visualization backend
using ProgressMeter      # Progress bar utilities for long computations
using LinearAlgebra      # Standard linear algebra operations (norm, dot, etc.)

  Activating project at `~/work/Horizons3DViewer2`


https://juliaastro.org/EphemerisSources.jl/docs/stable/examples/telescope-study/
https://juliaastro.org/AstroTime/stable/tutorial/#Tutorial
https://ssd.jpl.nasa.gov/horizons/app.html#/

In [11]:
######################################
# Specify the data to be retrieved
######################################

frame = "J2000"   # Input parameter for the NASA Horizons API
wrt   = "sun"     # Input parameter for the NASA Horizons API
units = "KM-S"    # Input parameter for the NASA Horizons API

####################
spacecraft = "Voyager 1"
start = TDBEpoch(1977, 09, 6, 0, 0, 0.0)
stop  = TDBEpoch(2026, 11, 01, 0, 0, 0.0)
step  = "3 day"
step_size_sec = 3*24*60*60


259200

In [12]:
############################################################################
# Retrieve ephemeris data and convert it into a convenient internal format
############################################################################

# Retrieve ephemeris data from JPL Horizons
sun_horizons     = ephemeris("sun", start, stop, step; wrt = wrt, units = units, frame = frame)
mercury_horizons = ephemeris("mercury", start, stop, step; wrt = wrt, units = units, frame = frame)
venus_horizons   = ephemeris("venus", start, stop, step; wrt = wrt, units = units, frame = frame)
earth_horizons   = ephemeris("earth", start, stop, step; wrt = wrt, units = units, frame = frame)
mars_horizons    = ephemeris("mars", start, stop, step; wrt = wrt, units = units, frame = frame)
jupiter_horizons = ephemeris("jupiter", start, stop, step; wrt = wrt, units = units, frame = frame)
saturn_horizons  = ephemeris("saturn", start, stop, step; wrt = wrt, units = units, frame = frame)
uranus_horizons  = ephemeris("uranus", start, stop, step; wrt = wrt, units = units, frame = frame)
neptune_horizons = ephemeris("neptune", start, stop, step; wrt = wrt, units = units, frame = frame)
pluto_horizons   = ephemeris("999", start, stop, step; wrt = wrt, units = units, frame = frame)

sc1_horizons     = ephemeris("Voyager 1", start, stop, step; wrt = wrt, units = units, frame = frame)
sc2_horizons     = ephemeris("Voyager 2", start, stop, step; wrt = wrt, units = units, frame = frame)

# Convert the Horizons outputs into 6×N state matrices
sun_states       = horizons_to_matrix(sun_horizons)
mercury_states   = horizons_to_matrix(mercury_horizons)
venus_states     = horizons_to_matrix(venus_horizons)
earth_states     = horizons_to_matrix(earth_horizons)
mars_states      = horizons_to_matrix(mars_horizons)
jupiter_states   = horizons_to_matrix(jupiter_horizons)
saturn_states    = horizons_to_matrix(saturn_horizons)
uranus_states    = horizons_to_matrix(uranus_horizons)
neptune_states   = horizons_to_matrix(neptune_horizons)
pluto_states     = horizons_to_matrix(pluto_horizons)

sc1_states       = horizons_to_matrix(sc1_horizons)
sc2_states       = horizons_to_matrix(sc2_horizons)

# Convert Horizons time stamps (Julian Date) to TDBEpoch values for animation labels
t_tdb = TDBEpoch.(sc1_horizons.t .* days; origin = :julian)

# Diagnostic output (for sanity checks)
println(size(sc1_states))
println(sc1_states[1:6, 1])
println(t_tdb[1])

(6, 5985)
[1.448005439108036e8, -4.220149282221936e7, 25598.45101304352, 11.57190972543096, 38.13748393606272, 0.7305166962567302]
1977-09-06T00:00:00.000 TDB


## v003: fixed planet orbits and moving markers

Run all cells from top to bottom in the existing project environment. The extra retrieval cell samples one approximate sidereal period per displayed body, starting at `start`, with about 1,000 points. Positions and labels still use the original animation time grid. Voyager trails and zoom settings are preserved. Orbit endpoints are not forced to close; perturbed ephemerides are not exactly periodic.

In [13]:
######################################
# Retrieve static, one-period planet orbits
######################################

# Approximate sidereal orbital periods in Julian years (365.25 days).
# Source: https://ssd.jpl.nasa.gov/planets/phys_par.html
# These are sampled real ephemerides, not exactly closed Keplerian ellipses.
orbit_samples = 1000
orbit_specs = [
    (name="Earth",   target="earth",   period_years=1.0000174, states=earth_states),
    (name="Mars",    target="mars",    period_years=1.8808476, states=mars_states),
    (name="Jupiter", target="jupiter", period_years=11.862615, states=jupiter_states),
    (name="Saturn",  target="saturn",  period_years=29.447498, states=saturn_states),
    (name="Uranus",  target="uranus",  period_years=84.016846, states=uranus_states),
    (name="Neptune", target="neptune", period_years=164.79132, states=neptune_states),
    (name="Pluto",   target="999",     period_years=247.92065, states=pluto_states),
]

function retrieve_planet_orbit(target, orbit_start, period_years; samples=1000, wrt="sun", units="KM-S", frame="J2000")
    samples >= 2 || throw(ArgumentError("samples must be at least 2"))
    period_days = period_years * 365.25
    # Integer-minute sampling gives approximately `samples` points per orbit.
    step_minutes = max(1, round(Int, period_days * 1440 / (samples - 1)))
    orbit_stop = orbit_start + period_days * days
    result = ephemeris(target, orbit_start, orbit_stop, "$(step_minutes) min"; wrt=wrt, units=units, frame=frame)
    states = horizons_to_matrix(result)
    size(states, 2) >= 2 || error("Too few orbit samples for $target")
    all(isfinite, states[1:3, :]) || error("Non-finite orbit positions for $target")
    return states
end

# Keep static orbit samples separate from the animation time grid.
planet_data = map(orbit_specs) do planet
    println("Retrieving one-period orbit: ", planet.name)
    orbit_start = start - (planet.period_years * 365.25 / 2) * days
    orbit_states = retrieve_planet_orbit(planet.target, orbit_start, planet.period_years; samples=orbit_samples, wrt=wrt, units=units, frame=frame)
    (name=planet.name, states=planet.states, orbit_states=orbit_states, color=:white)
end


Retrieving one-period orbit: Earth
Retrieving one-period orbit: Mars
Retrieving one-period orbit: Jupiter
Retrieving one-period orbit: Saturn
Retrieving one-period orbit: Uranus
Retrieving one-period orbit: Neptune
Retrieving one-period orbit: Pluto


7-element Vector{@NamedTuple{name::String, states::Matrix{Float64}, orbit_states::Matrix{Float64}, color::Symbol}}:
 (name = "Earth", states = [1.446562316472972e8 1.465235360922403e8 … 1.229379759162994e8 1.183098182151294e8; -4.255740267687239e7 -3.512615649048354e7 … 8.357920033423068e7 8.98184845297085e7; … ; 28.46481690959851 28.86270387374707 … 24.51576119174367 23.6157751619341; 0.001936296467532017 0.001252005756892771 … -0.001376924256843992 -0.0005690398177282674], orbit_states = [-1.446652248501398e8 -1.448892232972766e8 … -1.442220758300222e8 -1.444575927777553e8; 3.340695523243255e7 3.248717577430656e7 … 3.519409878656634e7 3.427627820627767e7; … ; -29.1225526657298 -29.16493804206299 … -29.05906447682634 -29.10425878150914; -0.00266528998463933 -0.002690921704775917 … -0.00068428582788016 -0.000609408108003251], color = :white)
 (name = "Mars", states = [1.316405010869627e8 1.26776299496615e8 … -3.518574289324103e7 -4.114674163521934e7; 1.781618168392779e8 1.8235409441828

In [14]:
######################################
# Animation settings
######################################

# Whether to render the animation (0: static image, 1: animation)
animation_flag = 1

# Length of the spacecraft trajectory trail
trail_len = 100000

# Animation frame selection
sampling_interval = 5
N = min(size(sc1_states, 2), size(sc2_states, 2), size(earth_states, 2), size(mars_states, 2), size(jupiter_states, 2), size(saturn_states, 2), size(uranus_states, 2), size(neptune_states, 2), size(pluto_states, 2), length(t_tdb))
frames = 1:sampling_interval:(N-1)

# Zoom settings (same units as the state coordinates: km)
zoom_min = 50e7
zoom_margin = 0.50

# Required half-width based on the distance of Voyager 1 and 2 from the Sun
zoom_radius1 = [sqrt(sum(abs2, sc1_states[1:3, i])) for i in 1:N]
zoom_radius2 = [sqrt(sum(abs2, sc2_states[1:3, i])) for i in 1:N]

# Never shrink the viewing range once expanded
zoom_extent = accumulate(max, max.(zoom_min, zoom_margin .* max.(zoom_radius1, zoom_radius2)))

######################################
# Plotting area settings
######################################

set_theme!(theme_black())
update_theme!(fontsize=18)

fig1 = Figure(size=(900, 600))
ax_ecliptic_s = default_axis3(fig1[1, 1]; scale=zoom_min, title="")

ax_ecliptic_s.aspect = :data
ax_ecliptic_s.viewmode = :fit

######################################
# Common plotting routines
######################################

# Sun
scatter!(ax_ecliptic_s, 0.0, 0.0, 0.0; markersize=15, color=:white)
text!(ax_ecliptic_s, 0.0, 0.0, 0.0; text="Sun", color=:white, fontsize=18, offset=(8, 8))

# Draw the complete planet orbits once; they remain fixed during animation.
for planet in planet_data
    lines!(ax_ecliptic_s, planet.orbit_states[1:3, :]; linewidth=0.5, linestyle=:dash, color=planet.color)
end

######################################
# Render static image or animation
######################################

if animation_flag == 0

    # Planet trajectories and labels at the last frame
    for planet in planet_data
        states = planet.states
        scatter!(ax_ecliptic_s, [Point3f(states[1, N], states[2, N], states[3, N])]; markersize=10, color=planet.color)
        text!(ax_ecliptic_s, Point3f(states[1, N], states[2, N], states[3, N]); text=planet.name, color=planet.color, fontsize=18, offset=(8, 8))
    end

    # Voyager trajectories
    lines!(ax_ecliptic_s, sc1_states[1:3, 1:N]; linewidth=1, color=:lime)
    lines!(ax_ecliptic_s, sc2_states[1:3, 1:N]; linewidth=1, color=:orange)

    text!(ax_ecliptic_s, Point3f(sc1_states[1, N], sc1_states[2, N], sc1_states[3, N]); text="Voyager 1", color=:lime, fontsize=18, offset=(8, 8))
    text!(ax_ecliptic_s, Point3f(sc2_states[1, N], sc2_states[2, N], sc2_states[3, N]); text="Voyager 2", color=:orange, fontsize=18, offset=(8, 8))

    # Fit all static orbit lines and both spacecraft trajectories.
    static_extent = maximum(maximum(abs, planet.orbit_states[1:3, :]) for planet in planet_data)
    spacecraft_extent = max(maximum(abs, sc1_states[1:3, 1:N]), maximum(abs, sc2_states[1:3, 1:N]))
    L = max(zoom_min, 1.10 * max(static_extent, spacecraft_extent))
    limits!(ax_ecliptic_s, -L, L, -L, L, -L, L)

else

    frame_0 = Observable(first(frames))

    # Initial viewing range
    L0 = zoom_extent[first(frames)]
    limits!(ax_ecliptic_s, -L0, L0, -L0, L0, -L0, L0)

    # Date
    text!(ax_ecliptic_s, 0.95, 0.85; text=@lift(Dates.format(t_tdb[$frame_0], "yyyy-mm-dd")), offset=(0, 0), align=(:right, :top), space=:relative)

    # Moving planet markers and labels (no fading planet trails)
    for planet in planet_data
        let states = planet.states, name = planet.name, color = planet.color
            position = @lift(Point3f(states[1, $frame_0], states[2, $frame_0], states[3, $frame_0]))
            text!(ax_ecliptic_s, position; text=name, color=color, fontsize=18, offset=(8, 8))
            scatter!(ax_ecliptic_s, @lift([$position]); markersize=10, color=color)
        end
    end

    # Voyager trajectories
    traj_fade!(ax_ecliptic_s, sc1_states[1:3, :], frame_0; trail_len=trail_len, color=:lime)
    traj_fade!(ax_ecliptic_s, sc2_states[1:3, :], frame_0; trail_len=trail_len, color=:orange)

    # Moving Voyager labels
    text!(ax_ecliptic_s, @lift(Point3f(sc1_states[1, $frame_0], sc1_states[2, $frame_0], sc1_states[3, $frame_0])); text="Voyager 1", color=:lime, fontsize=18, offset=(8, 8))
    text!(ax_ecliptic_s, @lift(Point3f(sc2_states[1, $frame_0], sc2_states[2, $frame_0], sc2_states[3, $frame_0])); text="Voyager 2", color=:orange, fontsize=18, offset=(8, 8))

    p = Progress(length(frames); desc="Rendering movie")

    # Export animation
    mkpath("../output")
    record(fig1, "../output/$(spacecraft)_v003.mp4", frames; framerate=30) do frame
        frame_0[] = frame

        # Zoom out as Voyager 1 and 2 move away from the Sun
        L = zoom_extent[frame]
        limits!(ax_ecliptic_s, -L, L, -L, L, -L, L)

        next!(p)
    end

end

display(fig1; inline=false)

Rendering movie 100%|████████████████████████████████████| Time: 0:00:37


GLMakie.Screen(...)